In [ ]:

"""
TASK 3: Deep-Dive Analysis & Interactive Dashboarding
ApexPlanet Software Pvt. Ltd. - Data Analytics Internship
Deep-Dive: Customer Segmentation + Cohort-style Retention Proxy
"""

import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

plt.rcParams.update({
    'figure.facecolor': '#0d1117', 'axes.facecolor': '#161b22',
    'axes.edgecolor': '#30363d', 'axes.labelcolor': '#e6edf3',
    'xtick.color': '#8b949e', 'ytick.color': '#8b949e',
    'text.color': '#e6edf3', 'grid.color': '#21262d',
    'grid.linestyle': '--', 'grid.alpha': 0.4,
})
ACCENT = ['#58a6ff','#3fb950','#f78166','#d2a8ff','#ffa657','#79c0ff','#56d364']

print("=" * 60)
print("TASK 3: Deep-Dive Analysis & Interactive Dashboarding")
print("=" * 60)

df = pd.read_csv('/content/sample_data/cleaned_dataset.csv', parse_dates=['transaction_date'])
print(f"Loaded: {df.shape}\n")

# ═══════════════════════════════════════════════════════════════
# STEP 1: DEFINE CORE KPIs
# ═══════════════════════════════════════════════════════════════
print("--- STEP 1: CORE KPIs ---")

total_txn      = len(df)
total_volume   = df['transaction_amount'].sum()
fraud_rate     = df['is_fraud'].mean() * 100
success_rate   = (df['transaction_status'] == 'Success').mean() * 100
avg_txn_value  = df['transaction_amount'].mean()
failed_count   = (df['transaction_status'].isin(['Failed','Reversed'])).sum()
churn_proxy    = (df['transaction_status'].isin(['Failed','Reversed'])).mean() * 100
loan_pct       = df['has_loan'].mean() * 100
digital_rate   = df['channel'].isin(['Mobile_App','Web','API']).mean() * 100

kpi_data = {
    'KPI': ['Total Transactions','Total Volume (₹)','Avg Transaction Value (₹)',
            'Transaction Success Rate','Fraud Rate','Failed+Reversed Rate',
            'Loan Penetration Rate','Digital Channel Rate'],
    'Formula': [
        'COUNT(transaction_id)',
        'SUM(transaction_amount)',
        'SUM(transaction_amount) / COUNT(transaction_id)',
        'COUNT(status=Success) / COUNT(*) × 100',
        'COUNT(is_fraud=1) / COUNT(*) × 100',
        'COUNT(status ∈ {Failed,Reversed}) / COUNT(*) × 100',
        'COUNT(has_loan=1) / COUNT(*) × 100',
        'COUNT(channel ∈ {Mobile,Web,API}) / COUNT(*) × 100'
    ],
    'Value': [
        f'{total_txn:,}', f'₹{total_volume:,.0f}', f'₹{avg_txn_value:,.2f}',
        f'{success_rate:.2f}%', f'{fraud_rate:.3f}%', f'{churn_proxy:.2f}%',
        f'{loan_pct:.2f}%', f'{digital_rate:.2f}%'
    ],
    'Business_Rationale': [
        'Scale of operations',
        'Total business processed',
        'Ticket size monitoring',
        'System reliability & customer trust',
        'Risk & compliance monitoring',
        'Customer experience failure rate',
        'Cross-sell / product penetration',
        'Digital transformation progress'
    ]
}
kpi_df = pd.DataFrame(kpi_data)
kpi_df.to_csv('/content/sample_data/core_kpis.csv', index=False)
print(kpi_df[['KPI','Value']].to_string(index=False))

# ═══════════════════════════════════════════════════════════════
# STEP 2: DEEP-DIVE — CUSTOMER SEGMENTATION
# ═══════════════════════════════════════════════════════════════
print("\n--- STEP 2: DEEP-DIVE — Customer Segmentation ---")

# Build customer-level features
cust = df.groupby('customer_id').agg(
    total_txn_count     = ('transaction_id','count'),
    total_spend         = ('transaction_amount', lambda x: x[df.loc[x.index,'transaction_direction']=='Debit'].sum()),
    total_receive       = ('transaction_amount', lambda x: x[df.loc[x.index,'transaction_direction']=='Credit'].sum()),
    avg_txn_amount      = ('transaction_amount','mean'),
    avg_balance         = ('account_balance','mean'),
    max_balance         = ('account_balance','max'),
    credit_score        = ('credit_score','first'),
    fraud_flag          = ('is_fraud','max'),
    has_loan            = ('has_loan','first'),
    months_active       = ('transaction_date', lambda x: x.dt.to_period('M').nunique()),
    unique_categories   = ('merchant_category','nunique'),
    failed_txns         = ('transaction_status', lambda x: (x.isin(['Failed','Reversed'])).sum()),
    account_type        = ('account_type','first'),
    state               = ('state','first'),
).reset_index()

cust['fail_rate'] = cust['failed_txns'] / cust['total_txn_count']
cust['net_flow']  = cust['total_receive'] - cust['total_spend']

print(f"Unique customers: {len(cust):,}")
print(f"Avg txns per customer: {cust['total_txn_count'].mean():.1f}")
print(f"Avg months active: {cust['months_active'].mean():.1f}")

# RFM-style segmentation using quantiles
cust['freq_score']  = pd.qcut(cust['total_txn_count'],   q=3, labels=[1,2,3]).astype(int)
cust['monetary_score'] = pd.qcut(cust['avg_txn_amount'], q=3, labels=[1,2,3]).astype(int)
cust['balance_score']  = pd.qcut(cust['avg_balance'],    q=3, labels=[1,2,3]).astype(int)
cust['rfm_total']   = cust['freq_score'] + cust['monetary_score'] + cust['balance_score']

def segment(score):
    if score >= 8: return 'Premium'
    elif score >= 6: return 'Active'
    elif score >= 4: return 'Regular'
    else: return 'Dormant'

cust['segment'] = cust['rfm_total'].apply(segment)
seg_counts = cust['segment'].value_counts()
print("\nCustomer Segments:")
print(seg_counts.to_string())

# ═══════════════════════════════════════════════════════════════
# STEP 3: COHORT-STYLE RETENTION PROXY (monthly activity)
# ═══════════════════════════════════════════════════════════════
print("\n--- STEP 3: MONTHLY COHORT RETENTION PROXY ---")

df['year_month'] = df['transaction_date'].dt.to_period('M')
first_month = df.groupby('customer_id')['year_month'].min().rename('cohort_month')
df2 = df.join(first_month, on='customer_id')

cohort_data = df2.groupby(['cohort_month','year_month'])['customer_id'].nunique().reset_index()
cohort_data.columns = ['cohort_month','activity_month','active_customers']
cohort_data['period_number'] = (cohort_data['activity_month'] - cohort_data['cohort_month']).apply(lambda x: x.n)

cohort_pivot = cohort_data.pivot_table(
    index='cohort_month', columns='period_number', values='active_customers')

# Limit to 12 periods for readability
cohort_pivot = cohort_pivot.iloc[:, :13]
cohort_size  = cohort_pivot[0]
retention    = cohort_pivot.divide(cohort_size, axis=0) * 100
retention_sample = retention.head(12).round(1)
print("\nRetention % (first 6 cohorts, 6 periods):")
print(retention_sample.iloc[:6, :7].to_string())

# ─── DASHBOARD FIGURE ────────────────────────────────────────
fig = plt.figure(figsize=(24, 16), facecolor='#0d1117')
gs  = gridspec.GridSpec(3, 4, figure=fig, hspace=0.45, wspace=0.35)
fig.suptitle('TASK 3 — Deep-Dive Dashboard: Indian Banking Analytics',
             fontsize=18, fontweight='bold', color='#58a6ff', y=0.98)

# — KPI tiles —
kpi_tiles = [
    (f'{total_txn/1000:.0f}K', 'Total Transactions', ACCENT[0]),
    (f'₹{total_volume/1e9:.1f}B', 'Total Volume', ACCENT[1]),
    (f'₹{avg_txn_value:,.0f}', 'Avg Txn Value', ACCENT[3]),
    (f'{success_rate:.1f}%', 'Success Rate', ACCENT[4]),
]
for i, (val, label, color) in enumerate(kpi_tiles):
    ax = fig.add_subplot(gs[0, i])
    ax.set_facecolor('#1c2128')
    ax.text(0.5, 0.60, val, ha='center', va='center', fontsize=22, fontweight='bold',
            color=color, transform=ax.transAxes)
    ax.text(0.5, 0.25, label, ha='center', va='center', fontsize=10,
            color='#8b949e', transform=ax.transAxes)
    ax.set_xticks([]); ax.set_yticks([])
    for spine in ax.spines.values():
        spine.set_edgecolor(color); spine.set_linewidth(2)

# — Customer segment donut —
ax_seg = fig.add_subplot(gs[1, 0])
wedges, texts, autos = ax_seg.pie(seg_counts.values, labels=seg_counts.index,
    autopct='%1.1f%%', colors=ACCENT[:4], startangle=90,
    wedgeprops={'edgecolor':'#0d1117','linewidth':2}, pctdistance=0.75)
for t in texts+autos:
    t.set_color('#e6edf3'); t.set_fontsize(9)
ax_seg.set_title('Customer Segments', color='#e6edf3', fontweight='bold')

# — Avg balance by segment —
ax_bal = fig.add_subplot(gs[1, 1])
seg_bal = cust.groupby('segment')['avg_balance'].mean().sort_values()
bars = ax_bal.barh(seg_bal.index, seg_bal.values/1000, color=ACCENT[:4], edgecolor='none')
ax_bal.set_title('Avg Balance by Segment (K₹)', color='#e6edf3', fontweight='bold')
ax_bal.set_xlabel('Avg Balance (₹K)')

# — Fraud rate by credit band —
ax_fraud = fig.add_subplot(gs[1, 2])
fraud_band = df.groupby('credit_score_band')['is_fraud'].mean().mul(100)
band_order = ['Excellent','Good','Fair','Poor','Very Poor']
fraud_band = fraud_band.reindex([b for b in band_order if b in fraud_band.index])
ax_fraud.bar(fraud_band.index, fraud_band.values,
             color=[ACCENT[1],ACCENT[0],ACCENT[4],ACCENT[2],ACCENT[2]], edgecolor='none')
ax_fraud.set_title('Fraud Rate by Credit Band (%)', color='#e6edf3', fontweight='bold')
ax_fraud.set_xlabel('Credit Score Band')
ax_fraud.set_ylabel('Fraud Rate (%)')
ax_fraud.tick_params(axis='x', rotation=30)

# — Monthly volume trend —
ax_trend = fig.add_subplot(gs[1, 3])
monthly = df.groupby(df['transaction_date'].dt.to_period('M'))['transaction_amount'].sum() / 1e6
monthly.index = monthly.index.to_timestamp()
ax_trend.plot(monthly.index, monthly.values, color=ACCENT[0], linewidth=2)
ax_trend.fill_between(monthly.index, monthly.values, alpha=0.12, color=ACCENT[0])
ax_trend.set_title('Monthly Volume (₹M)', color='#e6edf3', fontweight='bold')
ax_trend.tick_params(axis='x', rotation=45)
ax_trend.set_ylabel('Volume (₹M)')

# — Retention heatmap —
ax_ret = fig.add_subplot(gs[2, :3])
ret_plot = retention.head(18).iloc[:, :13].fillna(0)
sns.heatmap(ret_plot, annot=True, fmt='.0f', cmap='YlOrRd', ax=ax_ret,
            linewidths=0.3, cbar_kws={'label':'Retention %'},
            annot_kws={'size': 6})
ax_ret.set_title('Customer Cohort Retention % (Monthly)', color='#e6edf3', fontweight='bold')
ax_ret.set_xlabel('Months Since First Transaction')
ax_ret.set_ylabel('Cohort (Join Month)')
ax_ret.tick_params(axis='x', colors='#e6edf3', labelsize=8)
ax_ret.tick_params(axis='y', colors='#e6edf3', labelsize=7)

# — Digital vs Physical channel trend —
ax_dig = fig.add_subplot(gs[2, 3])
df['is_digital'] = df['channel'].isin(['Mobile_App','Web','API'])
digi_monthly = df.groupby(df['transaction_date'].dt.to_period('M'))['is_digital'].mean().mul(100)
digi_monthly.index = digi_monthly.index.to_timestamp()
ax_dig.plot(digi_monthly.index, digi_monthly.values, color=ACCENT[1], linewidth=2)
ax_dig.fill_between(digi_monthly.index, digi_monthly.values, alpha=0.12, color=ACCENT[1])
ax_dig.set_title('Digital Adoption Rate (%)', color='#e6edf3', fontweight='bold')
ax_dig.tick_params(axis='x', rotation=45)
ax_dig.set_ylabel('Digital %')

plt.savefig('/content/sample_data/fig3_dashboard.png', dpi=150, bbox_inches='tight',
            facecolor='#0d1117')
plt.close()
print("\n[FIG 3] Saved: fig3_dashboard.png")

# Save customer segments
cust[['customer_id','segment','total_txn_count','avg_txn_amount',
      'avg_balance','credit_score','has_loan','months_active','fail_rate']].to_csv(
    '/content/sample_data/customer_segments.csv', index=False)
print("[SAVED] customer_segments.csv")
kpi_df.to_csv('/content/sample_data/core_kpis.csv', index=False)
print("[SAVED] core_kpis.csv")
print("\n✅ Task 3 Complete!")

TASK 3: Deep-Dive Analysis & Interactive Dashboarding
Loaded: (550000, 32)

--- STEP 1: CORE KPIs ---
                      KPI           Value
       Total Transactions         550,000
         Total Volume (₹) ₹16,448,901,664
Avg Transaction Value (₹)      ₹29,907.09
 Transaction Success Rate          92.04%
               Fraud Rate          0.886%
     Failed+Reversed Rate           5.99%
    Loan Penetration Rate          34.96%
     Digital Channel Rate          66.91%

--- STEP 2: DEEP-DIVE — Customer Segmentation ---
Unique customers: 79,916
Avg txns per customer: 6.9
Avg months active: 6.5

Customer Segments:
segment
Active     33481
Regular    28664
Premium    11392
Dormant     6379

--- STEP 3: MONTHLY COHORT RETENTION PROXY ---

Retention % (first 6 cohorts, 6 periods):
period_number      0     1     2     3     4     5     6
cohort_month                                            
2019-01        100.0  10.0  11.4  10.3  10.9  11.2  11.3
2019-02        100.0  11.6  11.1  11